# Flow Configuration Testing

This notebook tests the FlowConfiguration model with different parameter sources and modules.

In [1]:
import sys
sys.path.append('..')

from ped.flow import FlowConfiguration
import json
from pprint import pprint

In [2]:
from ped.param.sources import ParameterSource

In [5]:
test_config = {
    "metadata": {
        "name": "Risk Assessment Flow",
        "description": "A flow for assessing customer risk using decision tables and inline calculations",
        "author": "Risk Team",
        "license": "MIT",
        "version": "1.0"
    },
    "parameter_sources": {
        "static_config": {
            "type": "static",
            "values": {
                "max_risk_score": 100,
                "default_threshold": 0.7,
                "min_credit_score": 650,
                "max_loan_amount": 500000,
                "interest_rates": {
                    "low_risk": 3.2,
                    "medium_risk": 4.5,
                    "high_risk": 6.8
                }
            }
        },
        "request_params": {
            "type": "inputs",
            "base_key": "parameters",
            "version_key": "version",
            "defaults": {
                "environment": "production",
                "debug_mode": False,
                "max_processing_time": 5000
            }
        }
    },

    "modules": {
        "risk_calculator": {
            "type": "inline",
            "test": "${ped.param:static_config,max_risk_score,test}",
            "test2": "${ped.param:request_params,max_processing_time}"
        }
    }
}

In [6]:
sample_request = {
    "parameters": {
        "environment": "staging",
        "debug_mode": True,
        "max_processing_time": 3000,
        "version": "v1.2"
    },
    "customer_data": {
        "age": 32,
        "income": 75000,
        "credit_score": 720
    }
}
flow_config = FlowConfiguration.model_validate(test_config)

parameterized_modules = await flow_config.get_parameterized_modules(inputs=sample_request)
parameterized_modules

{'risk_calculator': {'type': 'inline', 'test': 100, 'test2': 3000}}

## Test get_parameterized_modules()

Now let's test the parameterization functionality with sample request data.

In [ ]:
# Test with sample request data
sample_request = {
    "parameters": {
        "environment": "staging",
        "debug_mode": True,
        "max_processing_time": 3000,
        "version": "v1.2"
    },
    "customer_data": {
        "age": 32,
        "income": 75000,
        "credit_score": 720
    }
}

try:
    print("Testing get_parameterized_modules()...")
    parameterized_modules = flow_config.get_parameterized_modules(
        request=sample_request
    )
    
    print("✅ Parameterization successful!")
    print("\n=== Parameterized Modules ===")
    
    for module_name, module_config in parameterized_modules.items():
        print(f"\n--- {module_name} ---")
        print(f"Type: {module_config.get('type')}")
        
        if 'parameters' in module_config:
            print("Parameters:")
            for param_name, param_value in module_config['parameters'].items():
                print(f"  {param_name}: {param_value}")
        
        if 'input_mapping' in module_config:
            print(f"Input mapping: {module_config['input_mapping']}")
            
        if 'output_mapping' in module_config:
            print(f"Output mapping: {module_config['output_mapping']}")
            
except Exception as e:
    print(f"❌ Parameterization failed: {e}")
    import traceback
    traceback.print_exc()

## Test Different Request Scenarios

In [ ]:
# Test with minimal request
minimal_request = {}

print("Testing with minimal (empty) request...")
try:
    minimal_modules = flow_config.get_parameterized_modules(request=minimal_request)
    print("✅ Minimal request handled successfully")
    
    # Check if default values are used
    risk_calc_params = minimal_modules['risk_calculator']['parameters']
    print(f"Max risk score from static source: {risk_calc_params.get('max_risk_score')}")
    print(f"Threshold from static source: {risk_calc_params.get('threshold')}")
    
except Exception as e:
    print(f"❌ Minimal request failed: {e}")

In [ ]:
# Test with version-specific request
versioned_request = {
    "parameters": {
        "environment": "development",
        "debug_mode": True,
        "version": "v2.0"
    }
}

print("Testing with version-specific request...")
try:
    versioned_modules = flow_config.get_parameterized_modules(
        request=versioned_request,
        requested_versions={"request_params": "v2.0"}
    )
    print("✅ Versioned request handled successfully")
    
    # Show how parameters are resolved
    loan_decision_params = versioned_modules['loan_decision']['parameters']
    print(f"Environment from request: {loan_decision_params.get('environment')}")
    print(f"Min credit score from static: {loan_decision_params.get('min_credit_score')}")
    
except Exception as e:
    print(f"❌ Versioned request failed: {e}")

## Inspect Parameter Resolution Details

In [ ]:
# Let's examine how the parameter resolution works in detail
print("=== Parameter Resolution Analysis ===")

test_request = {
    "parameters": {
        "environment": "production",
        "debug_mode": False,
        "version": "v1.0"
    }
}

resolved_modules = flow_config.get_parameterized_modules(request=test_request)

print("\n--- Static Source Parameters ---")
static_source = flow_config.parameter_sources['static_config']
print(f"Type: {static_source.type}")
print(f"Available values: {list(static_source.values.keys())}")

print("\n--- Request Source Parameters ---")
request_source = flow_config.parameter_sources['request_params']
print(f"Type: {request_source.type}")
print(f"Base key: {request_source.base_key}")
print(f"Version key: {request_source.version_key}")
print(f"Defaults: {request_source.defaults}")

print("\n--- Module Parameter Resolution ---")
for module_name, module_config in resolved_modules.items():
    if 'parameters' in module_config:
        print(f"\n{module_name}:")
        for param_name, resolved_value in module_config['parameters'].items():
            print(f"  {param_name}: {resolved_value} (type: {type(resolved_value).__name__})")

## Test Edge Cases and Error Handling

In [ ]:
# Test configuration with missing parameter source reference
invalid_config = {
    "metadata": {
        "name": "Invalid Flow",
        "description": "This should fail validation",
        "author": "Test"
    },
    "parameter_sources": {
        "source1": {
            "type": "static",
            "values": {"key1": "value1"}
        }
    },
    "modules": {
        "module1": {
            "type": "inline",
            "module_config": {
                "expression_set": {"test": "parameter('missing_param')"}
            },
            "parameters": {
                "missing_param": {
                    "source": "parameter_source",
                    "source_name": "nonexistent_source",  # This should cause validation error
                    "key": "some_key"
                }
            }
        }
    }
}

print("Testing invalid configuration (should fail validation)...")
try:
    invalid_flow = FlowConfiguration.model_validate(invalid_config)
    print("❌ Expected validation to fail, but it passed!")
except Exception as e:
    print(f"✅ Validation correctly failed: {e}")

## Summary

This notebook demonstrates:

1. **Configuration Validation**: The FlowConfiguration model correctly validates complex nested configurations
2. **Parameter Sources**: Both static and request parameter sources work as expected
3. **Module Parameterization**: The `get_parameterized_modules()` method correctly resolves parameters from different sources
4. **Request Handling**: Different request scenarios (empty, versioned, etc.) are handled properly
5. **Error Handling**: Invalid configurations are caught and reported appropriately

The flow configuration system is working as designed!